# CI vs CD Grid -- Synthetic AR(1) Experiments

Trains PatchTST in channel-independent (CI) and channel-dependent (CD) modes
over a 3x3x2 grid: C in {7, 21, 84} x rho in {0.1, 0.5, 0.9} x {CI, CD}.

All hyperparameters are identical across CI and CD except batch size, which is
reduced for CD at large C to fit within T4 memory (see GRID_CONFIG comments).

The model uses d_model=64 and num_heads=8 (not the paper's 128/16) to reduce
training time. The effect being measured -- CI vs CD performance gap as a function
of C and rho -- is large enough to appear at reduced capacity. Results may shift
slightly at full capacity; this is acknowledged as a limitation in the paper.

Results are saved to `results/results_grid.csv` after every run. The notebook is
fully resumable: completed (C, rho, mode) combinations are skipped on rerun.

In [ ]:
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")


## Synthetic Dataset (inlined from synthetic/generate.py)

In [ ]:
# Inlined from time-series-forecasting/synthetic/generate.py.
# Python 3.10 compatibility: use Split = Literal[...] not type Split = ...

from typing import Literal

Split = Literal["train", "val", "test"]

_BURN_IN: int = 1000
_TRAIN_NUM: int = 8640   # ETTh1 split numerators over 14400 total rows
_VAL_NUM: int = 2880
_DENOM: int = 14400


def build_covariance(C: int, rho: float) -> np.ndarray:
    """Construct a compound-symmetry covariance matrix of shape (C, C).

    Positive definite for rho in (-1/(C-1), 1); singular at the boundaries.
    """
    if C < 2:
        raise ValueError(f"C must be >= 2; got {C}.")
    lower = -1.0 / (C - 1)
    if not (lower < rho < 1.0):
        raise ValueError(
            f"rho={rho} is outside the valid range ({lower:.6f}, 1.0) for C={C}."
        )
    return rho * np.ones((C, C), dtype=np.float64) + (1.0 - rho) * np.eye(C, dtype=np.float64)


def generate_ar1(T: int, C: int, phi: float, cov: np.ndarray, seed: int) -> np.ndarray:
    """Generate a multivariate AR(1) series and discard burn-in steps.

    Returns float64 array of shape (T - _BURN_IN, C).
    """
    if T <= _BURN_IN:
        raise ValueError(f"T={T} must be > _BURN_IN={_BURN_IN}.")
    rng = np.random.default_rng(seed)
    x = np.empty((T, C), dtype=np.float64)
    x[0] = rng.standard_normal(C)
    noise = rng.multivariate_normal(np.zeros(C), cov, size=T - 1)
    for t in range(1, T):
        x[t] = phi * x[t - 1] + noise[t - 1]
    return x[_BURN_IN:]


class SyntheticARDataset(Dataset):
    """Sliding-window dataset over a synthetic multivariate AR(1) series.

    Split protocol: 60/20/20 matching ETTh1 fractions.
    Normalization: StandardScaler fit on train split, applied to all splits.
    """

    def __init__(
        self, C: int, rho: float, phi: float, seq_len: int, pred_len: int,
        split: Split, seed: int, total_len: int = 14400,
    ) -> None:
        if split not in ("train", "val", "test"):
            raise ValueError(f"split must be one of 'train', 'val', 'test'; got '{split}'.")
        cov = build_covariance(C, rho)
        series = generate_ar1(total_len, C, phi, cov, seed)

        usable = len(series)
        train_end = usable * _TRAIN_NUM // _DENOM
        val_end   = train_end + usable * _VAL_NUM // _DENOM

        scaler = StandardScaler()
        scaler.fit(series[:train_end])
        normalized = scaler.transform(series).astype(np.float32)

        if split == "train":
            self._data = normalized[:train_end]
        elif split == "val":
            self._data = normalized[train_end:val_end]
        else:
            self._data = normalized[val_end:]

        window = seq_len + pred_len
        if len(self._data) < window:
            raise ValueError(
                f"Split '{split}' has {len(self._data)} rows but "
                f"seq_len + pred_len = {window}."
            )
        self.seq_len  = seq_len
        self.pred_len = pred_len

    def __len__(self) -> int:
        return len(self._data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx: int) -> tuple:
        x = self._data[idx : idx + self.seq_len]
        y = self._data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return torch.from_numpy(x), torch.from_numpy(y)


## Model (inlined from models/patchtst.py)

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, patch_size: int, stride: int, d_model: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.stride     = stride
        self.projection = nn.Linear(patch_size, d_model)
        self.dropout    = nn.Dropout(dropout)
        self._d_model   = d_model

    def _sinusoidal_pe(self, num_patches: int, device: torch.device) -> torch.Tensor:
        position = torch.arange(num_patches, device=device).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, self._d_model, 2, device=device) * (-math.log(10000.0) / self._d_model)
        )
        pe = torch.zeros(num_patches, self._d_model, device=device)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x  = x.squeeze(-1).unfold(dimension=-1, size=self.patch_size, step=self.stride)
        x  = self.projection(x)
        pe = self._sinusoidal_pe(x.shape[1], x.device)
        return self.dropout(x + pe)


class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normed    = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out
        return x + self.ff(self.norm2(x))


class TransformerEncoder(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_layers: int, dropout: float) -> None:
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, dropout) for _ in range(num_layers)])
        self.norm   = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)


class ForecastHead(nn.Module):
    def __init__(self, num_patches: int, d_model: int, pred_len: int, dropout: float) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.linear  = nn.Linear(num_patches * d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(self.dropout(x.flatten(1)))


class PatchTST(nn.Module):
    """PatchTST with CI and CD mode support.

    channel_mixing=False (CI): each variate processed independently.
    channel_mixing=True  (CD): patches from all variates concatenated before encoder.

    Reference: Nie et al., "A Time Series Is Worth 64 Words", ICLR 2023.
               https://arxiv.org/abs/2211.14730
    """

    def __init__(
        self, seq_len: int, pred_len: int, num_variates: int,
        patch_size: int = 16, stride: int = 8,
        d_model: int = 64, num_heads: int = 8, num_layers: int = 3,
        dropout: float = 0.2, channel_mixing: bool = False,
    ) -> None:
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads}).")
        self.num_variates   = num_variates
        self.channel_mixing = channel_mixing
        self.num_patches    = (seq_len - patch_size) // stride + 1
        self.embedding = PatchEmbedding(patch_size, stride, d_model, dropout)
        self.encoder   = TransformerEncoder(d_model, num_heads, num_layers, dropout)
        self.head      = ForecastHead(self.num_patches, d_model, pred_len, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, C = x.shape
        x = x.permute(0, 2, 1).reshape(B * C, L, 1)
        x = self.embedding(x)                           # (B*C, N, D)
        if self.channel_mixing:
            x = x.reshape(B, C * self.num_patches, -1)  # (B, C*N, D)
            x = self.encoder(x)
            x = x.reshape(B * C, self.num_patches, -1)  # (B*C, N, D)
        else:
            x = self.encoder(x)                         # (B*C, N, D)
        x = self.head(x)                                # (B*C, pred_len)
        return x.reshape(B, C, -1).permute(0, 2, 1)    # (B, pred_len, C)


## Training Infrastructure

In [ ]:
class EarlyStopping:
    def __init__(self, patience: int = 10, checkpoint_path: str = "best.pt") -> None:
        self.patience       = patience
        self.checkpoint_path = checkpoint_path
        self.best_val_mse   = float("inf")
        self.counter        = 0
        self.best_epoch     = 0

    def step(self, val_mse: float, model: nn.Module, epoch: int) -> bool:
        if val_mse < self.best_val_mse:
            self.best_val_mse = val_mse
            self.counter      = 0
            self.best_epoch   = epoch
            torch.save(model.state_dict(), self.checkpoint_path)
        else:
            self.counter += 1
        return self.counter >= self.patience


def compute_metrics(pred: torch.Tensor, target: torch.Tensor) -> tuple:
    return torch.mean((pred - target) ** 2).item(), torch.mean(torch.abs(pred - target)).item()


def train_one_epoch(model, loader, optimizer, criterion) -> tuple:
    model.train()
    total_mse, total_mae, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        _, mae = compute_metrics(pred.detach(), y)
        b = x.size(0)
        total_mse += loss.item() * b
        total_mae += mae * b
        n += b
    return total_mse / n, total_mae / n


@torch.no_grad()
def evaluate(model, loader) -> tuple:
    model.eval()
    total_mse, total_mae, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x)
        mse, mae = compute_metrics(pred, y)
        b = x.size(0)
        total_mse += mse * b
        total_mae += mae * b
        n += b
    return total_mse / n, total_mae / n


## Grid Config

In [ ]:
RESULTS_DIR = Path("results")
CKPT_DIR    = Path("results/checkpoints")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = RESULTS_DIR / "results_grid.csv"

# d_model=64, num_heads=8 (not paper's 128/16): intentionally smaller to reduce
# training time. The CI vs CD performance gap is large enough to appear at this
# capacity. The paper acknowledges results may shift slightly at full capacity.
GRID_CONFIG = {
    "seq_len":    512,
    "pred_len":   96,
    "patch_size": 16,
    "stride":     8,
    "d_model":    64,
    "num_heads":  8,
    "num_layers": 3,
    "dropout":    0.2,
    "lr":         1e-4,
    "warmup_epochs": 5,   # 10% of 50 epochs, identical for CI and CD
    "epochs":     50,
    "patience":   10,
    "seed":       42,
    "phi":        0.8,
    "total_len":  14400,
    # Batch size depends on (C, mode) to stay within T4 16 GB memory.
    # CD encoder receives C*N tokens (N=63 patches). Attention matrix scales as
    # B * H * (C*N)^2. CI processes each variate independently: memory scales
    # linearly with C, so batch_size=128 is safe for all C values.
    #
    # C=7:  CD tokens=441  -> batch_size=128 (attn ~0.8 GB at H=8)
    # C=21: CD tokens=1323 -> batch_size=32  (attn ~1.8 GB at H=8)
    # C=84: CD tokens=5292 -> batch_size=4   (attn ~3.6 GB at H=8)
    "batch_size_ci": 128,
    "batch_size_cd": {7: 128, 21: 32, 84: 4},
}

C_VALUES   = [7, 21, 84]
RHO_VALUES = [0.1, 0.5, 0.9]
MODES      = ["CI", "CD"]

print(f"Grid: {len(C_VALUES)} C values x {len(RHO_VALUES)} rho values x {len(MODES)} modes = "
      f"{len(C_VALUES) * len(RHO_VALUES) * len(MODES)} runs")


## Run Function

In [ ]:
def run_cell(C: int, rho: float, mode: str, config: dict, ckpt_dir: Path) -> dict:
    """Train one CI/CD run on synthetic AR(1) data and return a result dict.

    Args:
        C: Number of variates.
        rho: Off-diagonal correlation coefficient.
        mode: 'CI' or 'CD'.
        config: GRID_CONFIG dict.
        ckpt_dir: Directory for checkpoints.

    Returns:
        Dict with keys: dataset, C, rho, mode, pred_len, test_mse, test_mae,
        best_epoch, seed.
    """
    torch.cuda.empty_cache()
    channel_mixing = mode == "CD"
    seed           = config["seed"]
    batch_size     = (
        config["batch_size_ci"] if not channel_mixing
        else config["batch_size_cd"][C]
    )
    ckpt_path = str(ckpt_dir / f"synth_C{C}_rho{rho}_{mode.lower()}.pt")

    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    ds_kwargs = dict(C=C, rho=rho, phi=config["phi"], seq_len=config["seq_len"],
                     pred_len=config["pred_len"], seed=seed, total_len=config["total_len"])
    train_ds = SyntheticARDataset(split="train", **ds_kwargs)
    val_ds   = SyntheticARDataset(split="val",   **ds_kwargs)
    test_ds  = SyntheticARDataset(split="test",  **ds_kwargs)

    loader_kw    = {"num_workers": 2, "pin_memory": True}
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  **loader_kw)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **loader_kw)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **loader_kw)

    model = PatchTST(
        seq_len=config["seq_len"], pred_len=config["pred_len"], num_variates=C,
        patch_size=config["patch_size"], stride=config["stride"],
        d_model=config["d_model"], num_heads=config["num_heads"],
        num_layers=config["num_layers"], dropout=config["dropout"],
        channel_mixing=channel_mixing,
    ).to(DEVICE)

    total_params   = sum(p.numel() for p in model.parameters())
    warmup_epochs  = config["warmup_epochs"]

    print(f"[C={C} rho={rho} {mode}] params={total_params:,} | batch={batch_size} | "
          f"CD_tokens={C * model.num_patches if channel_mixing else 'N/A (CI)'}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=1e-4)

    def lr_lambda(epoch: int) -> float:
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, config["epochs"] - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler    = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    criterion    = nn.MSELoss()
    early_stop   = EarlyStopping(patience=config["patience"], checkpoint_path=ckpt_path)
    t0           = time.time()

    for epoch in range(1, config["epochs"] + 1):
        train_mse, _ = train_one_epoch(model, train_loader, optimizer, criterion)
        val_mse, _   = evaluate(model, val_loader)
        scheduler.step()
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{config['epochs']} | "
                  f"train {train_mse:.4f} | val {val_mse:.4f} | "
                  f"lr {optimizer.param_groups[0]['lr']:.2e} | {time.time()-t0:.0f}s")
        if early_stop.step(val_mse, model, epoch):
            print(f"  Early stop @ epoch {epoch}. Best: epoch {early_stop.best_epoch}.")
            break

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    test_mse, test_mae = evaluate(model, test_loader)
    print(f"  Test MSE: {test_mse:.4f} | MAE: {test_mae:.4f} | "
          f"Best val: {early_stop.best_val_mse:.4f} @ epoch {early_stop.best_epoch}")

    return {
        "dataset":   "synthetic_ar1",
        "C":         C,
        "rho":       rho,
        "mode":      mode,
        "pred_len":  config["pred_len"],
        "test_mse":  round(test_mse, 6),
        "test_mae":  round(test_mae, 6),
        "best_epoch": early_stop.best_epoch,
        "seed":      seed,
    }


## Verification: First Two Runs (C=7, rho=0.1)

In [ ]:
# Run C=7, rho=0.1 for CI and CD before launching the full grid.
# Verify shapes, loss decreases, and no OOM before committing to 18 runs.
print("=== Verification run: C=7, rho=0.1 ===")
verification_results = []
for mode in ["CI", "CD"]:
    print(f"\n--- {mode} ---")
    result = run_cell(C=7, rho=0.1, mode=mode, config=GRID_CONFIG, ckpt_dir=CKPT_DIR)
    verification_results.append(result)
    torch.cuda.empty_cache()

print("\nVerification complete. Results:")
for r in verification_results:
    print(f"  C={r['C']} rho={r['rho']} {r['mode']}: "
          f"test_mse={r['test_mse']:.4f} best_epoch={r['best_epoch']}")
print("\nIf both runs completed without error, proceed to the full grid.")


## Full Grid Run

In [ ]:
# Load existing results for resumability
if RESULTS_CSV.exists():
    existing_df = pd.read_csv(RESULTS_CSV)
    completed   = set(zip(existing_df["C"], existing_df["rho"], existing_df["mode"]))
    all_results = existing_df.to_dict("records")
    print(f"Resuming: {len(completed)} runs already complete, {18 - len(completed)} remaining.")
else:
    completed   = set()
    all_results = []
    print("Starting fresh grid run.")

# Add verification results if not already saved
for r in verification_results:
    key = (r["C"], r["rho"], r["mode"])
    if key not in completed:
        all_results.append(r)
        completed.add(key)

for C in C_VALUES:
    for rho in RHO_VALUES:
        for mode in MODES:
            if (C, rho, mode) in completed:
                print(f"[SKIP] C={C} rho={rho} {mode} already complete.")
                continue

            print(f'\n{"=" * 60}')
            print(f"C={C} | rho={rho} | mode={mode}")
            print(f'{"=" * 60}')
            result = run_cell(C=C, rho=rho, mode=mode, config=GRID_CONFIG, ckpt_dir=CKPT_DIR)
            all_results.append(result)
            completed.add((C, rho, mode))
            torch.cuda.empty_cache()

            # Save after every run so a timeout loses at most one result
            pd.DataFrame(all_results).to_csv(RESULTS_CSV, index=False)

# Final save and summary
results_df = pd.DataFrame(all_results)
results_df.to_csv(RESULTS_CSV, index=False)

print("\n=== Grid Results ===")
print(results_df[["C", "rho", "mode", "test_mse", "test_mae", "best_epoch"]].to_string(index=False))

print("\n--- MSE ratio (CD/CI) per (C, rho) cell ---")
pivot = results_df.pivot_table(index=["C", "rho"], columns="mode", values="test_mse")
pivot["ratio_cd_ci"] = pivot["CD"] / pivot["CI"]
pivot["winner"]      = np.where(pivot["CI"] < pivot["CD"], "CI", "CD")
print(pivot[["CI", "CD", "ratio_cd_ci", "winner"]].to_string())
print("\nNote: ratio > 1.0 means CI wins; ratio < 1.0 means CD wins.")
print("Results reported as observed; no sign assumed in advance.")


## Verify All 18 Rows Present

In [ ]:
if not RESULTS_CSV.exists():
    raise RuntimeError(f"{RESULTS_CSV} not found. Do not close the session.")

final_df = pd.read_csv(RESULTS_CSV)
expected = len(C_VALUES) * len(RHO_VALUES) * len(MODES)

if len(final_df) < expected:
    missing_keys = {
        (C, rho, mode)
        for C in C_VALUES for rho in RHO_VALUES for mode in MODES
    } - set(zip(final_df["C"], final_df["rho"], final_df["mode"]))
    raise RuntimeError(
        f"Expected {expected} rows in results_grid.csv; got {len(final_df)}. "
        f"Missing: {missing_keys}"
    )

print(f"All {expected} rows present in {RESULTS_CSV}.")
print(f"\nFinal CSV preview:")
print(final_df[["C", "rho", "mode", "test_mse", "test_mae", "best_epoch"]].to_string(index=False))
